# Data Preparation Pipelines for Knee Replacement Outcomes

## Key Workflow:
1. **Impute Q-question items** (pre-op and post-op questions 1-12)
2. **Calculate Q-Scores** (sum of items)
   - Pre-Op Q Score = Sum of Pre-Op Q questions 1-12
   - Post-Op Q Score = Sum of Post-Op Q questions 1-12
3. **Create target variable** = Post-Op Q Score - Pre-Op Q Score (improvement)
4. **Apply three imputation pipelines** to the prepared data

Each pipeline produces train/test datasets ready for modeling.

## Setup: Import Libraries

In [1]:
import logging
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Tuple, Dict, List
from datetime import datetime

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

print('✓ All libraries imported successfully')

✓ All libraries imported successfully


## Step 1: Load Data

In [11]:
# Define paths

# Load the Knee Replacement CCG data
# Load the Knee Replacement Provider data
df = pd.read_parquet(f'./data/interim/2.0-preprocessing.parquet')


# Make a copy of the original dataframe
df_original = df.copy()
print("Original dataframe copied to df_original.")
print(f"  Shape: {df_original.shape[0]:,} rows × {df_original.shape[1]} columns")
print(f"\nFirst few columns:")
print(df_original.columns[:].tolist())



Original dataframe copied to df_original.
  Shape: 139,236 rows × 78 columns

First few columns:
['Provider Code', 'Procedure', 'Revision Flag', 'Year', 'Age Band', 'Gender', 'Pre-Op Q Assisted', 'Pre-Op Q Assisted By', 'Pre-Op Q Symptom Period', 'Pre-Op Q Previous Surgery', 'Pre-Op Q Living Arrangements', 'Pre-Op Q Disability', 'Heart Disease', 'High Bp', 'Stroke', 'Circulation', 'Lung Disease', 'Diabetes', 'Kidney Disease', 'Nervous System', 'Liver Disease', 'Cancer', 'Depression', 'Arthritis', 'Pre-Op Q Mobility', 'Pre-Op Q Self-Care', 'Pre-Op Q Activity', 'Pre-Op Q Discomfort', 'Pre-Op Q Anxiety', 'Pre-Op Q EQ5D Index Profile', 'Pre-Op Q EQ5D Index', 'Post-Op Q Assisted', 'Post-Op Q Assisted By', 'Post-Op Q Living Arrangements', 'Post-Op Q Disability', 'Post-Op Q Mobility', 'Post-Op Q Self-Care', 'Post-Op Q Activity', 'Post-Op Q Discomfort', 'Post-Op Q Anxiety', 'Post-Op Q Satisfaction', 'Post-Op Q Sucess', 'Post-Op Q Allergy', 'Post-Op Q Bleeding', 'Post-Op Q Wound', 'Post-Op Q Ur

In [3]:
print("\n" + "=" * 70)
print("MISSINGNESS OVERVIEW (ALL COLUMNS)")
print("=" * 70)

n_rows = len(df_original)

missing_summary = []

for col in df_original.columns:
    missing_count = df_original[col].isnull().sum()
    if missing_count > 0:
        missing_pct = 100 * missing_count / n_rows
        missing_summary.append((col, missing_count, missing_pct))

# Sort by highest missing %
missing_summary = sorted(missing_summary, key=lambda x: x[2], reverse=True)

if missing_summary:
    print(f"\n✓ Columns with missing values:")
    for col, count, pct in missing_summary:
        print(f"  {col:50s} {count:10,d} ({pct:6.2f}%)")

    print(f"\nTotal columns with missing: {len(missing_summary)} / {len(df_original.columns)}")
else:
    print("✓ No missing values found")

# Optional: quick stats
total_missing = df.isnull().sum().sum()
total_cells = df_original.shape[0] * df_original.shape[1]
total_missing_pct = 100 * total_missing / total_cells

print(f"\n✓ Overall missingness:")
print(f"  Total missing values: {total_missing:,}")
print(f"  Percentage missing:   {total_missing_pct:.2f}%")


MISSINGNESS OVERVIEW (ALL COLUMNS)

✓ Columns with missing values:
  Post-Op Q Bleeding                                     17,138 ( 12.31%)
  Post-Op Q Urine                                        15,534 ( 11.16%)
  Post-Op Q Wound                                        14,740 ( 10.59%)
  Pre-Op Q EQ VAS                                        12,986 (  9.33%)
  Post-Op Q Allergy                                      12,779 (  9.18%)
  Age Band                                                9,402 (  6.75%)
  Gender                                                  9,402 (  6.75%)
  Pre-Op Q EQ5D Index                                     7,500 (  5.39%)
  Post-Op Q EQ VAS                                        6,571 (  4.72%)
  Post-Op Q EQ5D Index                                    6,274 (  4.51%)
  Pre-Op Q Disability                                     5,907 (  4.24%)
  Pre-Op Q Discomfort                                     5,540 (  3.98%)
  Knee Replacement Pre-Op Q Score           

## Step 2: Identify Q-Questions and Score Columns

In [4]:
# Identify Pre-Op Q-questions (items 1-12)
pre_op_q_items = [col for col in df_original.columns if col.startswith('Knee Replacement Pre-Op Q') and 'Score' not in col]
# Remove the composite scores for now
pre_op_q_items = [col for col in pre_op_q_items if col not in ['Knee Replacement Pre-Op Q Score', 'Knee Replacement Pre-Op Q EQ5D Index', 'Knee Replacement Pre-Op Q EQ VAS']]

# Identify Post-Op Q-questions (items 1-12)
post_op_q_items = [col for col in df_original.columns if col.startswith('Knee Replacement Post-Op Q') and 'Score' not in col]
# Remove the composite scores for now
post_op_q_items = [col for col in post_op_q_items if col not in ['Knee Replacement Post-Op Q Score', 'Knee Replacement Post-Op Q EQ5D Index', 'Knee Replacement Post-Op Q EQ VAS']]

print(f"✓ Pre-Op Q-items (questions 1-12): {len(pre_op_q_items)}")
for col in sorted(pre_op_q_items):
    print(f"  {col}")

print(f"\n✓ Post-Op Q-items (questions 1-12): {len(post_op_q_items)}")
for col in sorted(post_op_q_items):
    print(f"  {col}")

✓ Pre-Op Q-items (questions 1-12): 12
  Knee Replacement Pre-Op Q Confidence
  Knee Replacement Pre-Op Q Kneeling
  Knee Replacement Pre-Op Q Limping
  Knee Replacement Pre-Op Q Night Pain
  Knee Replacement Pre-Op Q Pain
  Knee Replacement Pre-Op Q Shopping
  Knee Replacement Pre-Op Q Stairs
  Knee Replacement Pre-Op Q Standing
  Knee Replacement Pre-Op Q Transport
  Knee Replacement Pre-Op Q Walking
  Knee Replacement Pre-Op Q Washing
  Knee Replacement Pre-Op Q Work

✓ Post-Op Q-items (questions 1-12): 12
  Knee Replacement Post-Op Q Confidence
  Knee Replacement Post-Op Q Kneeling
  Knee Replacement Post-Op Q Limping
  Knee Replacement Post-Op Q Night Pain
  Knee Replacement Post-Op Q Pain
  Knee Replacement Post-Op Q Shopping
  Knee Replacement Post-Op Q Stairs
  Knee Replacement Post-Op Q Standing
  Knee Replacement Post-Op Q Transport
  Knee Replacement Post-Op Q Walking
  Knee Replacement Post-Op Q Washing
  Knee Replacement Post-Op Q Work


## Step 3: Prepare Data for Imputation

We'll impute the individual Q-questions first, then calculate the scores.

In [5]:
# Create a working copy
df_work = df_original.copy()

# Identify all feature columns (exclude metadata and original scores)
exclude_cols = ['has_missing', 'Knee Replacement Pre-Op Q Score', 'Knee Replacement Post-Op Q Score']
feature_cols = [col for col in df_work.columns if col not in exclude_cols]

# Separate numeric and categorical
numeric_cols = df_work[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df_work[feature_cols].select_dtypes(include=['object', 'category']).columns.tolist()

print(f"Feature columns:")
print(f"  Numeric: {len(numeric_cols)}")
print(f"  Categorical: {len(categorical_cols)}")
print(f"  Total: {len(feature_cols)}")

# Extract the Q-question items from feature columns
q_items_to_impute = [col for col in pre_op_q_items + post_op_q_items if col in feature_cols]
print(f"\n✓ Q-items to impute: {len(q_items_to_impute)}")
print(f"  Pre-Op Q-items: {len([col for col in pre_op_q_items if col in feature_cols])} items")
print(f"  Post-Op Q-items: {len([col for col in post_op_q_items if col in feature_cols])} items")

# Other numeric features (not Q-items)
other_numeric = [col for col in numeric_cols if col not in q_items_to_impute]
print(f"✓ Other numeric features: {len(other_numeric)}")

print(f"\n⚠️  Note: Post-Op Q-items will be imputed to calculate the target variable,")
print(f"    but will be EXCLUDED from features (to prevent data leakage)")


Feature columns:
  Numeric: 76
  Categorical: 0
  Total: 76

✓ Q-items to impute: 24
  Pre-Op Q-items: 12 items
  Post-Op Q-items: 12 items
✓ Other numeric features: 52

⚠️  Note: Post-Op Q-items will be imputed to calculate the target variable,
    but will be EXCLUDED from features (to prevent data leakage)


## Step 4: Create Target Variable (Q-Score Difference)

Calculate the difference between post-op and pre-op Q-scores as the target.

In [6]:
# Identify columns with >2% missingness (for missingness indicators in Pipelines 2 & 3)
print("\n" + "=" * 70)
print("IDENTIFYING COLUMNS WITH HIGH MISSINGNESS (>2%)")
print("=" * 70)

# Exclude post-op Q-items, score columns, and derived outcome measures
# from the missingness-indicator analysis only
cols_to_exclude_from_indicators = post_op_q_items.copy()
cols_to_exclude_from_indicators += [
    'Pre-Op Q Score Calculated',
    'Post-Op Q Score Calculated',
    'Knee Replacement Pre-Op Q Score',
    'Knee Replacement Post-Op Q Score',
    'Pre-Op Q EQ5D Index Profile',
    'Pre-Op Q EQ5D Index',
    'Pre-Op Q EQ VAS'
]

# Remove duplicates while preserving order
cols_to_exclude_from_indicators = list(dict.fromkeys(cols_to_exclude_from_indicators))

# Retained columns for missingness analysis
feature_cols_analysis = [
    col for col in feature_cols
    if col not in cols_to_exclude_from_indicators
]

missing_pcts = {}

# Build analysis dataset safely
if 'y_original' in globals():
    # Align target to df_work index first
    y_aligned = y_original.reindex(df_work.index)
    valid_target_mask = y_aligned.notnull()

    df_analysis = df_work.loc[valid_target_mask, feature_cols_analysis].copy()
    print("\n✓ Using rows with valid target only (aligned to df_work.index)")
else:
    df_analysis = df_work.loc[:, feature_cols_analysis].copy()
    print("\n✓ y_original not found - using all rows")

n_rows = len(df_analysis)

for col in feature_cols_analysis:
    missing_pct = 100 * df_analysis[col].isnull().sum() / n_rows
    missing_pcts[col] = missing_pct

# Columns that should get a missingness indicator
cols_for_missingness_indicators = [
    col for col, pct in missing_pcts.items()
    if pct > 2
]

# Sort for reporting only
cols_with_high_missingness = sorted(
    [(col, missing_pcts[col]) for col in cols_for_missingness_indicators],
    key=lambda x: x[1],
    reverse=True
)

print("\n✓ Columns retained for analysis:")
print(f"  {len(feature_cols_analysis)} columns")

print("\n✓ Columns with >2% missingness (these get missingness indicators):")
if cols_with_high_missingness:
    for col, pct in cols_with_high_missingness:
        print(f"  {col:50s} {pct:6.2f}%")
    print(f"\nTotal columns getting indicators: {len(cols_for_missingness_indicators)}")
else:
    print("  None found")

print("\n✓ Columns retained WITHOUT missingness indicators:")
cols_without_missingness_indicators = [
    col for col in feature_cols_analysis
    if col not in cols_for_missingness_indicators
]
print(f"  {len(cols_without_missingness_indicators)} columns")

print("\n✓ Summary:")
print(f"  Retained columns total:                 {len(feature_cols_analysis)}")
print(f"  Columns with missingness indicators:    {len(cols_for_missingness_indicators)}")
print(f"  Columns without indicators:             {len(cols_without_missingness_indicators)}")

# Create missingness indicator columns in df_work
for col in cols_for_missingness_indicators:
    df_work[f"{col}_missing"] = df_work[col].isnull().astype(int)

print(f"\n✓ Added {len(cols_for_missingness_indicators)} missingness indicator columns to df_work")


IDENTIFYING COLUMNS WITH HIGH MISSINGNESS (>2%)

✓ y_original not found - using all rows

✓ Columns retained for analysis:
  61 columns

✓ Columns with >2% missingness (these get missingness indicators):
  Post-Op Q Bleeding                                  12.31%
  Post-Op Q Urine                                     11.16%
  Post-Op Q Wound                                     10.59%
  Post-Op Q Allergy                                    9.18%
  Age Band                                             6.75%
  Gender                                               6.75%
  Post-Op Q EQ VAS                                     4.72%
  Post-Op Q EQ5D Index                                 4.51%
  Pre-Op Q Disability                                  4.24%
  Pre-Op Q Discomfort                                  3.98%
  Post-Op Q Disability                                 3.76%
  Pre-Op Q Anxiety                                     3.71%
  Pre-Op Q Activity                                    3.22%
  

In [7]:
# Identify columns with >2% missingness (for missingness indicators in pipelines 2 & 3)
print("\n" + "="*70)
print("IDENTIFYING COLUMNS WITH HIGH MISSINGNESS (>2%)")
print("="*70)

# Calculate missingness percentage for all feature columns 
# Exclude post-op Q-items, score columns, and derived outcome measures 
# (these will be removed from final feature set)
cols_to_exclude_from_indicators = post_op_q_items.copy()
cols_to_exclude_from_indicators += ['Pre-Op Q Score Calculated', 'Post-Op Q Score Calculated',
                                     'Knee Replacement Pre-Op Q Score', 'Knee Replacement Post-Op Q Score']
cols_to_exclude_from_indicators += ['Pre-Op Q EQ5D Index Profile', 'Pre-Op Q EQ5D Index', 'Pre-Op Q EQ VAS']

feature_cols_analysis = [col for col in feature_cols if col not in cols_to_exclude_from_indicators]
missing_pcts = {}

for col in feature_cols_analysis:
    missing_pct = 100 * df_work[col].isnull().sum() / len(df_work)
    if missing_pct > 2:
        missing_pcts[col] = missing_pct

# Sort by missingness percentage
cols_with_high_missingness = sorted(missing_pcts.items(), key=lambda x: x[1], reverse=True)

print(f"\n✓ Columns with >2% missingness (in feature set, excluding post-op, score, and derived outcome columns):")
if cols_with_high_missingness:
    for col, pct in cols_with_high_missingness:
        print(f"  {col:50s} {pct:6.2f}%")
    print(f"\nTotal: {len(cols_with_high_missingness)} columns")
else:
    print("  None found")

# Store for use in pipelines
cols_for_missingness_indicators = [col for col, _ in cols_with_high_missingness]
print(f"\n✓ These columns will have missingness indicators in Pipelines 2 & 3")


IDENTIFYING COLUMNS WITH HIGH MISSINGNESS (>2%)

✓ Columns with >2% missingness (in feature set, excluding post-op, score, and derived outcome columns):
  Post-Op Q Bleeding                                  12.31%
  Post-Op Q Urine                                     11.16%
  Post-Op Q Wound                                     10.59%
  Post-Op Q Allergy                                    9.18%
  Age Band                                             6.75%
  Gender                                               6.75%
  Post-Op Q EQ VAS                                     4.72%
  Post-Op Q EQ5D Index                                 4.51%
  Pre-Op Q Disability                                  4.24%
  Pre-Op Q Discomfort                                  3.98%
  Post-Op Q Disability                                 3.76%
  Pre-Op Q Anxiety                                     3.71%
  Pre-Op Q Activity                                    3.22%
  Pre-Op Q Self-Care                                 

## Step 5: Imputation Pipelines

Now we implement the three pipelines with the correct workflow:
1. Use only rows with valid targets
2. Impute Q-question items
3. Calculate the Q-scores from imputed items
4. Create target as the difference
5. Use other features as predictors

### Pipeline 1: Missing Data Preservation

In [8]:
print("\n" + "=" * 70)
print("DIAGNOSTIC: MISSINGNESS BEFORE vs AFTER EXCLUSION")
print("=" * 70)

# Build a row mask aligned to df_work
if 'y_original' in globals():
    y_aligned = y_original.reindex(df_work.index)
    valid_target_mask = y_aligned.notna()
    df_analysis = df_work.loc[valid_target_mask, :].copy()
    print("\n✓ Using rows with valid target only (aligned to df_work.index)")
else:
    df_analysis = df_work.copy()
    print("\n✓ y_original not found - using all rows")

n_rows = len(df_analysis)

# Check missingness in the full feature set (before exclusion)
print(f"\n✓ Columns with missing values (in FULL feature set):")
full_missing = {}

for col in feature_cols:
    missing_count = df_analysis[col].isnull().sum()
    if missing_count > 0:
        missing_pct = 100 * missing_count / n_rows
        full_missing[col] = (missing_count, missing_pct)

if full_missing:
    for col, (count, pct) in sorted(full_missing.items(), key=lambda x: x[1][1], reverse=True):
        print(f"  {col:50s} {count:6,d} ({pct:5.2f}%)")
    print(f"\nTotal columns with missing: {len(full_missing)}")
else:
    print("  None found")

# Check which missing columns are being excluded
print(f"\n✓ Of these, how many are being EXCLUDED:")
excluded_with_missing = {
    col: (count, pct)
    for col, (count, pct) in full_missing.items()
    if col in cols_to_exclude_from_indicators
}

if excluded_with_missing:
    for col, (count, pct) in sorted(excluded_with_missing.items(), key=lambda x: x[1][1], reverse=True):
        print(f"  {col:50s} {count:6,d} ({pct:5.2f}%)")
    print(f"\nTotal excluded columns with missing: {len(excluded_with_missing)}")
else:
    print("  None (all missing values are in retained features)")

# Check missingness in retained features
print(f"\n✓ Missing values in RETAINED features (after exclusion):")
retained_with_missing = {
    col: (count, pct)
    for col, (count, pct) in full_missing.items()
    if col not in cols_to_exclude_from_indicators
}

if retained_with_missing:
    for col, (count, pct) in sorted(retained_with_missing.items(), key=lambda x: x[1][1], reverse=True):
        print(f"  {col:50s} {count:6,d} ({pct:5.2f}%)")
    print(f"\nTotal retained columns with missing: {len(retained_with_missing)}")
else:
    print("  None (all missing data was in excluded columns)")

# Of the retained columns, identify which should get missingness indicators (>2%)
print(f"\n✓ Retained columns with >2% missingness (will get indicators):")
indicator_with_missing = {
    col: (count, pct)
    for col, (count, pct) in retained_with_missing.items()
    if pct > 2
}

if indicator_with_missing:
    for col, (count, pct) in sorted(indicator_with_missing.items(), key=lambda x: x[1][1], reverse=True):
        print(f"  {col:50s} {count:6,d} ({pct:5.2f}%)")
    print(f"\nTotal retained columns getting indicators: {len(indicator_with_missing)}")
else:
    print("  None")

# Store for later pipeline use
cols_for_missingness_indicators = list(indicator_with_missing.keys())

# Retained columns that stay in the model but do NOT get indicators
retained_without_indicators = [
    col for col in feature_cols
    if col not in cols_to_exclude_from_indicators and col not in cols_for_missingness_indicators
]

print(f"\n✓ Summary:")
print(f"  Total feature columns:                         {len(feature_cols)}")
print(f"  Columns excluded:                              {len(cols_to_exclude_from_indicators)}")
print(f"  Columns retained:                              {len(feature_cols) - len(cols_to_exclude_from_indicators)}")
print(f"  Columns with missing in full feature set:      {len(full_missing)}")
print(f"  Columns with missing that are excluded:        {len(excluded_with_missing)}")
print(f"  Columns with missing that are retained:        {len(retained_with_missing)}")
print(f"  Retained columns getting indicators (>2%):     {len(cols_for_missingness_indicators)}")
print(f"  Retained columns without indicators:           {len(retained_without_indicators)}")


DIAGNOSTIC: MISSINGNESS BEFORE vs AFTER EXCLUSION

✓ y_original not found - using all rows

✓ Columns with missing values (in FULL feature set):
  Post-Op Q Bleeding                                 17,138 (12.31%)
  Post-Op Q Urine                                    15,534 (11.16%)
  Post-Op Q Wound                                    14,740 (10.59%)
  Pre-Op Q EQ VAS                                    12,986 ( 9.33%)
  Post-Op Q Allergy                                  12,779 ( 9.18%)
  Age Band                                            9,402 ( 6.75%)
  Gender                                              9,402 ( 6.75%)
  Pre-Op Q EQ5D Index                                 7,500 ( 5.39%)
  Post-Op Q EQ VAS                                    6,571 ( 4.72%)
  Post-Op Q EQ5D Index                                6,274 ( 4.51%)
  Pre-Op Q Disability                                 5,907 ( 4.24%)
  Pre-Op Q Discomfort                                 5,540 ( 3.98%)
  Post-Op Q Disability    

In [12]:
# ============================================================
# DEFINE TARGET
# ============================================================

target_col = "Knee Replacement Post-Op Q Score"   # <-- replace with your actual target column

y_original = df_work[target_col].copy()

print(f"✓ y_original created from: {target_col}")
print(f"  Non-missing target rows: {y_original.notna().sum():,}")
print(f"  Missing target rows:     {y_original.isna().sum():,}")

✓ y_original created from: Knee Replacement Post-Op Q Score
  Non-missing target rows: 136,276
  Missing target rows:     2,960


In [13]:
# ============================================================
# PIPELINE 1: KEEP RETAINED FEATURES, PRESERVE MISSING VALUES
# ============================================================

target_col = "YOUR_TARGET_COLUMN_NAME"   # <-- replace with your actual target column

# Create y_original if it does not exist yet
if 'y_original' not in globals():
    y_original = df_work[target_col].copy()
    print(f"\n✓ y_original created from: {target_col}")

# For Pipeline 1:
# - Keep all retained baseline/pre-op features
# - Preserve missing values as-is
# - Exclude all post-op variables and leakage / derived outcome columns
# - Do NOT add missingness indicators in Pipeline 1

# Build exclusion list
cols_to_exclude = post_op_q_items.copy()

# Add any other columns with "post" in the name (case-insensitive)
cols_to_exclude += [
    col for col in feature_cols
    if 'post' in col.lower() and col not in cols_to_exclude
]

# Add score columns that directly determine / strongly leak the target
cols_to_exclude += [
    'Pre-Op Q Score Calculated',
    'Post-Op Q Score Calculated',
    'Knee Replacement Pre-Op Q Score',
    'Knee Replacement Post-Op Q Score'
]

# Add derived / outcome-related columns that should not be used as baseline predictors
cols_to_exclude += [
    'Pre-Op Q EQ5D Index Profile',
    'Pre-Op Q EQ5D Index',
    'Pre-Op Q EQ VAS'
]

# Remove duplicates while preserving order
cols_to_exclude = list(dict.fromkeys(cols_to_exclude))

# Final retained feature set for Pipeline 1
feature_cols_p1 = [
    col for col in feature_cols
    if col not in cols_to_exclude
]

# Build aligned target mask the same way as in your diagnostic block
y_aligned = y_original.reindex(df_work.index)
valid_target_mask = y_aligned.notna()

X_p1 = df_work.loc[valid_target_mask, feature_cols_p1].copy()
y_p1 = y_aligned.loc[valid_target_mask].copy()

print("\n" + "=" * 70)
print("PIPELINE 1: FEATURES WITH MISSING VALUES PRESERVED")
print("=" * 70)

print(f"\n✓ Excluded {len(cols_to_exclude)} columns:")
print("  - All post-op variables")
print("  - Pre-op and post-op score columns (to prevent target leakage)")
print("  - Derived outcome measures (EQ5D indices, EQ VAS)")

print(f"\n✓ Retained features for Pipeline 1: {len(feature_cols_p1)}")

print(f"\n✓ Rows used for Pipeline 1:")
print(f"  {len(X_p1):,} rows with non-missing target")

print(f"\n✓ Missing values in Pipeline 1:")
missing_count = X_p1.isnull().sum().sum()
missing_pct = 100 * missing_count / (X_p1.shape[0] * X_p1.shape[1])
print(f"  Total missing: {missing_count:,} ({missing_pct:.2f}%)")

# Optional diagnostic: which retained columns have >2% missingness
cols_for_missingness_indicators_p1 = [
    col for col in feature_cols_p1
    if 100 * X_p1[col].isnull().sum() / len(X_p1) > 2
]

print(f"\n✓ Retained columns with >2% missingness:")
if cols_for_missingness_indicators_p1:
    for col in sorted(
        cols_for_missingness_indicators_p1,
        key=lambda c: X_p1[c].isnull().mean(),
        reverse=True
    ):
        pct = 100 * X_p1[col].isnull().sum() / len(X_p1)
        print(f"  {col:50s} {pct:6.2f}%")
else:
    print("  None")

print(f"\n✓ Note:")
print("  Pipeline 1 keeps these missing values as-is.")
print("  Missingness indicators are only for Pipelines 2 & 3.")

# Train-test split
X_train_p1, X_test_p1, y_train_p1, y_test_p1 = train_test_split(
    X_p1, y_p1, test_size=0.2, random_state=42
)

print(f"\n✓ Train-test split:")
print(f"  Train: {X_train_p1.shape[0]:,} samples")
print(f"  Test: {X_test_p1.shape[0]:,} samples")
print(f"  Features: {X_train_p1.shape[1]}")


PIPELINE 1: FEATURES WITH MISSING VALUES PRESERVED

✓ Excluded 39 columns:
  - All post-op variables
  - Pre-op and post-op score columns (to prevent target leakage)
  - Derived outcome measures (EQ5D indices, EQ VAS)

✓ Retained features for Pipeline 1: 41

✓ Rows used for Pipeline 1:
  136,276 rows with non-missing target

✓ Missing values in Pipeline 1:
  Total missing: 66,661 (1.19%)

✓ Retained columns with >2% missingness:
  Age Band                                             6.75%
  Gender                                               6.75%
  Pre-Op Q Disability                                  4.23%
  Pre-Op Q Discomfort                                  3.96%
  Pre-Op Q Anxiety                                     3.68%
  Pre-Op Q Activity                                    3.21%
  Pre-Op Q Self-Care                                   3.15%
  Pre-Op Q Mobility                                    3.06%

✓ Note:
  Pipeline 1 keeps these missing values as-is.
  Missingness indicato

In [14]:
# Save Pipeline 1
output_dir = Path.cwd() / 'prepared_datasets'
output_dir.mkdir(parents=True, exist_ok=True)

X_train_p1.to_parquet(output_dir / "pipeline1_missing_train.parquet")
X_test_p1.to_parquet(output_dir / "pipeline1_missing_test.parquet")
y_train_p1.to_frame(name='Q_Score_Difference').to_parquet(output_dir / "pipeline1_missing_train_target.parquet")
y_test_p1.to_frame(name='Q_Score_Difference').to_parquet(output_dir / "pipeline1_missing_test_target.parquet")

print(f"✓ Pipeline 1 saved to {output_dir}")

✓ Pipeline 1 saved to c:\Users\GinoH\OneDrive - SGB SMIT Group\Documents\GitHub\EAISI-NHS\notebooks\Experiment\Knee\prepared_datasets


### Pipeline 2: Median/Mode Imputation (Impute Q-items, then calculate scores)

In [16]:
print("\n" + "=" * 70)
print("PIPELINE 2: MEDIAN/MODE IMPUTATION")
print("=" * 70)

# Use same rows as Pipeline 1
if 'y_original' in globals():
    y_aligned = y_original.reindex(df_work.index)
    valid_target_mask = y_aligned.notna()
    df_p2 = df_work.loc[valid_target_mask].copy()
    y_p2 = y_aligned.loc[valid_target_mask].copy()
    print(f"\n✓ Starting with {len(df_p2):,} valid rows")
else:
    raise NameError("y_original is not defined. Define y_original before running Pipeline 2.")

# Create missingness indicators BEFORE train-test split
print(f"\n✓ Creating missingness indicators for {len(cols_for_missingness_indicators)} high-missingness columns...")
for col in cols_for_missingness_indicators:
    df_p2[f'{col}_is_missing'] = df_p2[col].isnull().astype(int)
    print(f"  Created: {col}_is_missing")

# Train-test split FIRST (before imputation to prevent data leakage)
df_train_p2, df_test_p2, y_train_p2, y_test_p2 = train_test_split(
    df_p2, y_p2, test_size=0.2, random_state=42
)

print(f"\n✓ Train-test split (before imputation):")
print(f"  Train: {len(df_train_p2):,} samples")
print(f"  Test: {len(df_test_p2):,} samples")



PIPELINE 2: MEDIAN/MODE IMPUTATION

✓ Starting with 136,276 valid rows

✓ Creating missingness indicators for 20 high-missingness columns...
  Created: Age Band_is_missing
  Created: Gender_is_missing
  Created: Pre-Op Q Disability_is_missing
  Created: Pre-Op Q Mobility_is_missing
  Created: Pre-Op Q Self-Care_is_missing
  Created: Pre-Op Q Activity_is_missing
  Created: Pre-Op Q Discomfort_is_missing
  Created: Pre-Op Q Anxiety_is_missing
  Created: Post-Op Q Living Arrangements_is_missing
  Created: Post-Op Q Disability_is_missing
  Created: Post-Op Q Mobility_is_missing
  Created: Post-Op Q Activity_is_missing
  Created: Post-Op Q Discomfort_is_missing
  Created: Post-Op Q Anxiety_is_missing
  Created: Post-Op Q Allergy_is_missing
  Created: Post-Op Q Bleeding_is_missing
  Created: Post-Op Q Wound_is_missing
  Created: Post-Op Q Urine_is_missing
  Created: Post-Op Q EQ5D Index_is_missing
  Created: Post-Op Q EQ VAS_is_missing

✓ Train-test split (before imputation):
  Train: 109,0

### Pipeline 2: Median/Mode Imputation with Missingness Indicators

We'll add binary indicator columns for features with >2% missingness before imputation.


In [17]:
# Step 1: Impute the Q-question items
print(f"\n✓ Imputing Q-question items...")

# Numeric imputer for Q-items
q_items_numeric = [col for col in q_items_to_impute if col in numeric_cols]
numeric_imputer = SimpleImputer(strategy='median')

# Fit on training data only
df_train_p2[q_items_numeric] = numeric_imputer.fit_transform(df_train_p2[q_items_numeric])
df_test_p2[q_items_numeric] = numeric_imputer.transform(df_test_p2[q_items_numeric])

print(f"  Imputed {len(q_items_numeric)} numeric Q-items with median")

# Categorical imputer for any categorical Q-items
q_items_categorical = [col for col in q_items_to_impute if col in categorical_cols]
if q_items_categorical:
    categorical_imputer = SimpleImputer(strategy='most_frequent')
    df_train_p2[q_items_categorical] = categorical_imputer.fit_transform(df_train_p2[q_items_categorical])
    df_test_p2[q_items_categorical] = categorical_imputer.transform(df_test_p2[q_items_categorical])
    print(f"  Imputed {len(q_items_categorical)} categorical Q-items with mode")

# Impute other numeric features
other_numeric_imputer = SimpleImputer(strategy='median')
df_train_p2[other_numeric] = other_numeric_imputer.fit_transform(df_train_p2[other_numeric])
df_test_p2[other_numeric] = other_numeric_imputer.transform(df_test_p2[other_numeric])

print(f"  Imputed {len(other_numeric)} other numeric features with median")

# Impute categorical features
if categorical_cols:
    cat_imputer = SimpleImputer(strategy='most_frequent')
    df_train_p2[categorical_cols] = cat_imputer.fit_transform(df_train_p2[categorical_cols])
    df_test_p2[categorical_cols] = cat_imputer.transform(df_test_p2[categorical_cols])
    print(f"  Imputed {len(categorical_cols)} categorical features with mode")


✓ Imputing Q-question items...
  Imputed 24 numeric Q-items with median
  Imputed 52 other numeric features with median


In [18]:
# Step 2: Recalculate Q-scores from imputed Q-items
print(f"\n✓ Recalculating Q-scores from imputed items...")

# Pre-Op Q Score = sum of pre-op Q-items
pre_op_q_items_in_data = [col for col in pre_op_q_items if col in df_train_p2.columns]
df_train_p2['Pre-Op Q Score Calculated'] = df_train_p2[pre_op_q_items_in_data].sum(axis=1)
df_test_p2['Pre-Op Q Score Calculated'] = df_test_p2[pre_op_q_items_in_data].sum(axis=1)

# Post-Op Q Score = sum of post-op Q-items
post_op_q_items_in_data = [col for col in post_op_q_items if col in df_train_p2.columns]
df_train_p2['Post-Op Q Score Calculated'] = df_train_p2[post_op_q_items_in_data].sum(axis=1)
df_test_p2['Post-Op Q Score Calculated'] = df_test_p2[post_op_q_items_in_data].sum(axis=1)

print(f"  Pre-Op Q Score (calculated): mean = {df_train_p2['Pre-Op Q Score Calculated'].mean():.2f}")
print(f"  Post-Op Q Score (calculated): mean = {df_train_p2['Post-Op Q Score Calculated'].mean():.2f}")

# Recalculate target = Post-Op - Pre-Op
y_train_p2_recalc = df_train_p2['Post-Op Q Score Calculated'] - df_train_p2['Pre-Op Q Score Calculated']
y_test_p2_recalc = df_test_p2['Post-Op Q Score Calculated'] - df_test_p2['Pre-Op Q Score Calculated']

print(f"\n✓ Recalculated target variable:")
print(f"  Train mean difference: {y_train_p2_recalc.mean():.2f}")
print(f"  Test mean difference: {y_test_p2_recalc.mean():.2f}")


✓ Recalculating Q-scores from imputed items...
  Pre-Op Q Score (calculated): mean = 19.02
  Post-Op Q Score (calculated): mean = 35.97

✓ Recalculated target variable:
  Train mean difference: 16.95
  Test mean difference: 16.87


In [19]:
# Step 3: Prepare features (exclude all post-op variables and calculated score columns)
# NOTE: ALL columns with "post" in the name are excluded to prevent data leakage
# NOTE: Pre-op and post-op score columns are excluded (they determine the target)
# NOTE: Additional derived outcome measures excluded (not baseline patient characteristics)
# NOTE: Missingness indicators are included (created before imputation)
cols_to_exclude = post_op_q_items.copy()
# Add any other columns with "post" in the name (case-insensitive)
cols_to_exclude += [col for col in feature_cols if 'post' in col.lower() and col not in cols_to_exclude]
# Add pre-op and post-op score columns (these directly determine the target)
cols_to_exclude += ['Pre-Op Q Score Calculated', 'Post-Op Q Score Calculated',
                    'Knee Replacement Pre-Op Q Score', 'Knee Replacement Post-Op Q Score']
# Add additional columns that represent predicted/derived outcomes (not baseline patient info)
cols_to_exclude += ['Pre-Op Q EQ5D Index Profile', 'Pre-Op Q EQ5D Index', 'Pre-Op Q EQ VAS']

feature_cols_for_model = [col for col in feature_cols if col not in cols_to_exclude]

# Add missingness indicator columns (but exclude indicators for the above columns)
missingness_indicator_cols = [f'{col}_is_missing' for col in cols_for_missingness_indicators 
                              if col not in cols_to_exclude]
feature_cols_for_model_with_indicators = feature_cols_for_model + missingness_indicator_cols

X_train_p2 = df_train_p2[feature_cols_for_model_with_indicators].copy()
X_test_p2 = df_test_p2[feature_cols_for_model_with_indicators].copy()

print(f"✓ Feature set prepared:")
print(f"  Base features (all post-op, score, and derived outcome columns excluded): {len(feature_cols_for_model)}")
print(f"  Missingness indicators: {len(missingness_indicator_cols)}")

✓ Feature set prepared:
  Base features (all post-op, score, and derived outcome columns excluded): 41
  Missingness indicators: 8


In [20]:
# Save Pipeline 2
X_train_p2.to_parquet(output_dir / "pipeline2_median_mode_train.parquet")
X_test_p2.to_parquet(output_dir / "pipeline2_median_mode_test.parquet")
y_train_p2_recalc.to_frame(name='Q_Score_Difference').to_parquet(output_dir / "pipeline2_median_mode_train_target.parquet")
y_test_p2_recalc.to_frame(name='Q_Score_Difference').to_parquet(output_dir / "pipeline2_median_mode_test_target.parquet")

print(f"✓ Pipeline 2 saved")

✓ Pipeline 2 saved


### Pipeline 3: MICE Imputation with Missingness Indicators

We'll add binary indicator columns for features with >2% missingness before imputation.


In [22]:
print("\n" + "=" * 70)
print("PIPELINE 3: MICE IMPUTATION")
print("=" * 70)

# Use same rows as others (aligned mask)
if 'y_original' in globals():
    y_aligned = y_original.reindex(df_work.index)
    valid_target_mask = y_aligned.notna()
    df_p3 = df_work.loc[valid_target_mask].copy()
    y_p3 = y_aligned.loc[valid_target_mask].copy()
    print(f"\n✓ Starting with {len(df_p3):,} valid rows")
else:
    raise NameError("y_original is not defined. Define y_original first.")

# Create missingness indicators BEFORE train-test split
print(f"\n✓ Creating missingness indicators for {len(cols_for_missingness_indicators)} high-missingness columns...")
for col in cols_for_missingness_indicators:
    df_p3[f'{col}_is_missing'] = df_p3[col].isnull().astype(int)
    print(f"  Created: {col}_is_missing")

# Train-test split
df_train_p3, df_test_p3, y_train_p3, y_test_p3 = train_test_split(
    df_p3, y_p3, test_size=0.2, random_state=42
)

print(f"\n✓ Train-test split:")
print(f"  Train: {len(df_train_p3):,} samples")
print(f"  Test: {len(df_test_p3):,} samples")



PIPELINE 3: MICE IMPUTATION

✓ Starting with 136,276 valid rows

✓ Creating missingness indicators for 20 high-missingness columns...
  Created: Age Band_is_missing
  Created: Gender_is_missing
  Created: Pre-Op Q Disability_is_missing
  Created: Pre-Op Q Mobility_is_missing
  Created: Pre-Op Q Self-Care_is_missing
  Created: Pre-Op Q Activity_is_missing
  Created: Pre-Op Q Discomfort_is_missing
  Created: Pre-Op Q Anxiety_is_missing
  Created: Post-Op Q Living Arrangements_is_missing
  Created: Post-Op Q Disability_is_missing
  Created: Post-Op Q Mobility_is_missing
  Created: Post-Op Q Activity_is_missing
  Created: Post-Op Q Discomfort_is_missing
  Created: Post-Op Q Anxiety_is_missing
  Created: Post-Op Q Allergy_is_missing
  Created: Post-Op Q Bleeding_is_missing
  Created: Post-Op Q Wound_is_missing
  Created: Post-Op Q Urine_is_missing
  Created: Post-Op Q EQ5D Index_is_missing
  Created: Post-Op Q EQ VAS_is_missing

✓ Train-test split:
  Train: 109,020 samples
  Test: 27,256 s

In [23]:
# Encode categorical features (MICE needs numeric)
print(f"\n✓ Encoding categorical features...")

df_train_p3_enc = df_train_p3.copy()
df_test_p3_enc = df_test_p3.copy()
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    all_values = pd.concat([df_train_p3[col], df_test_p3[col]], ignore_index=True).astype(str)
    le.fit(all_values)
    
    df_train_p3_enc[col] = le.transform(df_train_p3[col].astype(str))
    df_test_p3_enc[col] = le.transform(df_test_p3[col].astype(str))
    label_encoders[col] = le

print(f"  Encoded {len(categorical_cols)} categorical features")


✓ Encoding categorical features...
  Encoded 0 categorical features


In [24]:
# Fit MICE on all features (to capture relationships)
print(f"\n✓ Fitting IterativeImputer (MICE)...")
print(f"  This may take 1-2 minutes...\n")

mice_imputer = IterativeImputer(
    max_iter=10,
    random_state=42,
    verbose=0
)

# Fit and transform
X_train_p3_array = mice_imputer.fit_transform(df_train_p3_enc[feature_cols])
X_test_p3_array = mice_imputer.transform(df_test_p3_enc[feature_cols])

print(f"✓ MICE imputation complete!")


✓ Fitting IterativeImputer (MICE)...
  This may take 1-2 minutes...

✓ MICE imputation complete!


In [25]:
# Convert back to DataFrames
df_train_p3_imputed = pd.DataFrame(X_train_p3_array, columns=feature_cols, index=df_train_p3.index)
df_test_p3_imputed = pd.DataFrame(X_test_p3_array, columns=feature_cols, index=df_test_p3.index)

# Decode categorical features
print(f"\n✓ Decoding categorical features...")

for col in categorical_cols:
    le = label_encoders[col]
    
    # Round and clip to valid range
    train_encoded = df_train_p3_imputed[col].round().astype(int).clip(0, len(le.classes_)-1)
    test_encoded = df_test_p3_imputed[col].round().astype(int).clip(0, len(le.classes_)-1)
    
    # Inverse transform
    df_train_p3_imputed[col] = le.inverse_transform(train_encoded)
    df_test_p3_imputed[col] = le.inverse_transform(test_encoded)

print(f"  Decoded successfully")


✓ Decoding categorical features...
  Decoded successfully


In [26]:
# Recalculate Q-scores from imputed items
print(f"\n✓ Recalculating Q-scores from MICE-imputed items...")

pre_op_q_items_in_data = [col for col in pre_op_q_items if col in df_train_p3_imputed.columns]
post_op_q_items_in_data = [col for col in post_op_q_items if col in df_train_p3_imputed.columns]

df_train_p3_imputed['Pre-Op Q Score Calculated'] = df_train_p3_imputed[pre_op_q_items_in_data].sum(axis=1)
df_test_p3_imputed['Pre-Op Q Score Calculated'] = df_test_p3_imputed[pre_op_q_items_in_data].sum(axis=1)

df_train_p3_imputed['Post-Op Q Score Calculated'] = df_train_p3_imputed[post_op_q_items_in_data].sum(axis=1)
df_test_p3_imputed['Post-Op Q Score Calculated'] = df_test_p3_imputed[post_op_q_items_in_data].sum(axis=1)

# Target = difference
y_train_p3_recalc = df_train_p3_imputed['Post-Op Q Score Calculated'] - df_train_p3_imputed['Pre-Op Q Score Calculated']
y_test_p3_recalc = df_test_p3_imputed['Post-Op Q Score Calculated'] - df_test_p3_imputed['Pre-Op Q Score Calculated']

print(f"  Train mean difference: {y_train_p3_recalc.mean():.2f}")
print(f"  Test mean difference: {y_test_p3_recalc.mean():.2f}")


✓ Recalculating Q-scores from MICE-imputed items...
  Train mean difference: 16.95
  Test mean difference: 16.87


In [27]:
# Prepare features (exclude all post-op variables and score columns)
# NOTE: ALL columns with "post" in the name are excluded to prevent data leakage
# NOTE: Pre-op and post-op score columns are excluded (they determine the target)
# NOTE: Additional derived outcome measures excluded (not baseline patient characteristics)
# NOTE: Missingness indicators are included (created before imputation)
cols_to_exclude = post_op_q_items.copy()
# Add any other columns with "post" in the name (case-insensitive)
cols_to_exclude += [col for col in feature_cols if 'post' in col.lower() and col not in cols_to_exclude]
# Add pre-op and post-op score columns (these directly determine the target)
cols_to_exclude += ['Pre-Op Q Score Calculated', 'Post-Op Q Score Calculated',
                    'Knee Replacement Pre-Op Q Score', 'Knee Replacement Post-Op Q Score']
# Add additional columns that represent predicted/derived outcomes (not baseline patient info)
cols_to_exclude += ['Pre-Op Q EQ5D Index Profile', 'Pre-Op Q EQ5D Index', 'Pre-Op Q EQ VAS']

feature_cols_for_model = [col for col in feature_cols if col not in cols_to_exclude]

# Add missingness indicator columns (but exclude indicators for the above columns)
missingness_indicator_cols = [f'{col}_is_missing' for col in cols_for_missingness_indicators 
                              if col not in cols_to_exclude]
feature_cols_for_model_with_indicators = feature_cols_for_model + missingness_indicator_cols

X_train_p3 = df_train_p3_imputed[feature_cols_for_model].copy()
X_test_p3 = df_test_p3_imputed[feature_cols_for_model].copy()

# Add missingness indicators from the original (non-encoded) dataframes
for col in cols_for_missingness_indicators:
    if col not in cols_to_exclude:
        X_train_p3[f'{col}_is_missing'] = df_train_p3[f'{col}_is_missing']
        X_test_p3[f'{col}_is_missing'] = df_test_p3[f'{col}_is_missing']

print(f"✓ Feature set prepared:")
print(f"  Base features (all post-op, score, and derived outcome columns excluded): {len(feature_cols_for_model)}")

✓ Feature set prepared:
  Base features (all post-op, score, and derived outcome columns excluded): 41


In [28]:
# Save Pipeline 3
X_train_p3.to_parquet(output_dir / "pipeline3_mice_train.parquet")
X_test_p3.to_parquet(output_dir / "pipeline3_mice_test.parquet")
y_train_p3_recalc.to_frame(name='Q_Score_Difference').to_parquet(output_dir / "pipeline3_mice_train_target.parquet")
y_test_p3_recalc.to_frame(name='Q_Score_Difference').to_parquet(output_dir / "pipeline3_mice_test_target.parquet")

print(f"✓ Pipeline 3 saved")

✓ Pipeline 3 saved


In [34]:
pd.set_option('display.max_columns', None)  # show all columns
pd.set_option('display.width', None)        # prevent line wrapping

X_train_p1.head()  # or .head(10)

,Provider Code,Procedure,Revision Flag,Year,Age Band,Gender,Pre-Op Q Assisted,Pre-Op Q Assisted By,Pre-Op Q Symptom Period,Pre-Op Q Previous Surgery,Pre-Op Q Living Arrangements,Pre-Op Q Disability,Heart Disease,High Bp,Stroke,Circulation,Lung Disease,Diabetes,Kidney Disease,Nervous System,Liver Disease,Cancer,Depression,Arthritis,Pre-Op Q Mobility,Pre-Op Q Self-Care,Pre-Op Q Activity,Pre-Op Q Discomfort,Pre-Op Q Anxiety,Knee Replacement Pre-Op Q Pain,Knee Replacement Pre-Op Q Night Pain,Knee Replacement Pre-Op Q Washing,Knee Replacement Pre-Op Q Transport,Knee Replacement Pre-Op Q Walking,Knee Replacement Pre-Op Q Standing,Knee Replacement Pre-Op Q Limping,Knee Replacement Pre-Op Q Kneeling,Knee Replacement Pre-Op Q Work,Knee Replacement Pre-Op Q Confidence,Knee Replacement Pre-Op Q Shopping,Knee Replacement Pre-Op Q Stairs
72298,210,1,0,2,4.0,2.0,2.0,1,4.0,2.0,1.0,1.0,0,0,0,0,0,0,0,0,0,0,0,1,2.0,1.0,2.0,2.0,1.0,1.0,1.0,4.0,1.0,2.0,2.0,1.0,0.0,2.0,2.0,0.0,1.0
99591,34,1,0,3,4.0,1.0,2.0,1,3.0,2.0,1.0,2.0,0,0,0,1,0,0,0,0,0,0,0,1,2.0,2.0,2.0,3.0,1.0,0.0,0.0,2.0,2.0,2.0,1.0,1.0,1.0,1.0,1.0,0.0,2.0
82938,257,1,0,2,3.0,2.0,2.0,1,2.0,2.0,1.0,2.0,0,0,0,0,0,0,0,0,0,0,0,0,2.0,1.0,1.0,2.0,1.0,1.0,2.0,4.0,3.0,4.0,2.0,1.0,3.0,3.0,3.0,4.0,4.0
43805,276,1,0,1,3.0,1.0,2.0,1,2.0,2.0,2.0,2.0,0,1,0,0,0,0,0,0,0,0,0,1,2.0,1.0,2.0,2.0,1.0,1.0,0.0,3.0,3.0,2.0,1.0,2.0,1.0,1.0,2.0,3.0,2.0
133224,266,1,0,3,5.0,2.0,1.0,1,3.0,2.0,2.0,1.0,0,0,0,0,0,0,0,0,0,0,1,1,2.0,2.0,3.0,3.0,2.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,2.0,0.0,1.0


## Summary: All Pipelines Complete

In [29]:
print("\n" + "="*70)
print("ALL PIPELINES COMPLETE")
print("="*70)

# Create comparison
comparison = {
    'Pipeline': ['Pipeline 1 (Missing)', 'Pipeline 2 (Median/Mode)', 'Pipeline 3 (MICE)'],
    'Train Samples': [X_train_p1.shape[0], X_train_p2.shape[0], X_train_p3.shape[0]],
    'Test Samples': [X_test_p1.shape[0], X_test_p2.shape[0], X_test_p3.shape[0]],
    'Features': [X_train_p1.shape[1], X_train_p2.shape[1], X_train_p3.shape[1]],
    'Missing Values (Train)': [
        X_train_p1.isnull().sum().sum(),
        X_train_p2.isnull().sum().sum(),
        X_train_p3.isnull().sum().sum()
    ],
    'Target Mean (Train)': [
        y_train_p1.mean(),
        y_train_p2_recalc.mean(),
        y_train_p3_recalc.mean()
    ]
}

comparison_df = pd.DataFrame(comparison)
print("\n" + comparison_df.to_string(index=False))
print("\n" + "="*70)


ALL PIPELINES COMPLETE

                Pipeline  Train Samples  Test Samples  Features  Missing Values (Train)  Target Mean (Train)
    Pipeline 1 (Missing)         109020         27256        41                   53336            35.969593
Pipeline 2 (Median/Mode)         109020         27256        49                       0            16.946588
       Pipeline 3 (MICE)         109020         27256        49                       0            16.947844



## Output Files Generated

In [30]:
# List all generated files
print(f"\nGenerated files in: {output_dir}\n")

output_files = sorted(output_dir.glob('*.parquet'))
for i, f in enumerate(output_files, 1):
    size_kb = f.stat().st_size / 1024
    print(f"{i:2d}. {f.name:50s} ({size_kb:8.1f} KB)")

print(f"\nTotal: {len(output_files)} files")
total_size_mb = sum(f.stat().st_size for f in output_files) / (1024*1024)
print(f"Total size: {total_size_mb:.2f} MB")


Generated files in: c:\Users\GinoH\OneDrive - SGB SMIT Group\Documents\GitHub\EAISI-NHS\notebooks\Experiment\Knee\prepared_datasets

 1. pipeline1_missing_test.parquet                     (   491.0 KB)
 2. pipeline1_missing_test_target.parquet              (   195.6 KB)
 3. pipeline1_missing_train.parquet                    (  1890.9 KB)
 4. pipeline1_missing_train_target.parquet             (   792.2 KB)
 5. pipeline2_median_mode_test.parquet                 (   490.0 KB)
 6. pipeline2_median_mode_test_target.parquet          (   199.1 KB)
 7. pipeline2_median_mode_train.parquet                (  1870.2 KB)
 8. pipeline2_median_mode_train_target.parquet         (   805.8 KB)
 9. pipeline3_mice_test.parquet                        (   748.3 KB)
10. pipeline3_mice_test_target.parquet                 (   208.2 KB)
11. pipeline3_mice_train.parquet                       (  3011.8 KB)
12. pipeline3_mice_train_target.parquet                (   859.4 KB)

Total: 12 files
Total size: 11.29 MB


### Pipeline Descriptions

**Pipeline 1: Missing Data Preservation**
- No imputation
- Missing values retained as-is (for tree-based models)
- No missingness indicators

**Pipeline 2: Median/Mode Imputation**
- SimpleImputer with median (numeric) and mode (categorical)
- Imputes all missing values
- **Includes missingness indicator columns** for features with >2% missingness
- Indicators capture the original missing pattern for modeling

**Pipeline 3: MICE Imputation**
- IterativeImputer with Bayesian Ridge (10 iterations)
- Imputes based on relationships between features
- **Includes missingness indicator columns** for features with >2% missingness
- Indicators capture the original missing pattern for modeling

### Imputation Workflow
1. **Identify Q-items** (pre-op and post-op questions)
2. **Create missingness indicators** (Pipelines 2 & 3 only, for columns with >2% missing)
3. **Impute Q-items** (both pre-op AND post-op, needed to calculate target)
4. **Impute other features** (all remaining missing values)
5. **Recalculate Q-scores** by summing the imputed items
6. **Create target** as the difference between post and pre scores
7. **Remove ALL post-op variables and score columns** (data leakage prevention)
8. **Prepare features** for modeling:
   - **INCLUDE:** Pre-Op features (individual items only, NOT pre-op score) + other non-post features + missingness indicators
   - **EXCLUDE:** ALL post-op variables + pre-op/post-op score columns (data leakage prevention)

### Why Exclude ALL Post-Op Variables AND Score Columns?
- **Post-op variables:** Directly determine the target (post-op Q-Score is the numerator of the difference)
- **Pre-op Q-Score:** Used to calculate the target (is the denominator of the difference)
- **Post-op Q-Score:** The primary component of the target variable
- Including ANY of these means the model has direct access to what it's trying to predict
- This is **severe data leakage** - the model would achieve artificially perfect accuracy
- The individual pre-op Q-items are retained since they represent patient baseline characteristics
- At prediction time, we CANNOT have post-op information (it occurs after surgery)
- At prediction time, we CANNOT have pre-op Q-Score (only individual item responses are available)
- Therefore, models must learn from pre-op item responses and patient characteristics only

### Data Leakage Prevention
- Train-test split is done **before** imputation AND before creating indicators
- Imputers are fit **only on training data**
- Test data uses imputation parameters from training
- **ALL post-op variables AND score columns are removed** before any model training
- Missingness indicators are created from original data before imputation
- This ensures realistic model evaluation and future applicability
- At prediction time on new patients, only pre-op item responses will be available

### Examples of Excluded Variables
**Post-Op Variables:**
- Post-Op Q-items (Q1-Q12 post-op questions)
- Post-Op Q Score
- Any other columns with "post" in the name (case-insensitive)

**Score Columns:**
- Pre-Op Q Score (calculated from individual items during preprocessing)
- Post-Op Q Score (calculated from individual items during preprocessing)
- Knee Replacement Pre-Op Q Score (original data)
- Knee Replacement Post-Op Q Score (original data)
